# Runs saved in another mouse's MED-PC file

For per-animal MED-PC files, the published conversion matched a run by start date and time only and took the first run listed at that time. Where another mouse's run was saved in the file at the same start time and listed first, that run was converted under the file's mouse. The fixed code keeps only the runs whose `Subject:` field refers to the file's mouse (following the lab's answer in `seiler_2024_notes.md`: trust the subject number and disregard misfiled runs) and matches each run by its box.

This notebook reconverts the two sessions whose own runs were never converted and lists the published sessions that held another mouse's run and are no longer converted.

In [1]:
from datetime import datetime
from pathlib import Path
import shutil

import h5py
import pandas as pd

from lerner_lab_to_nwb.seiler_2024.medpc_helpers import get_medpc_variables
from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_dataset import (
    fp_to_nwb,
    opto_to_nwb,
    get_no_port_entry_duration_sessions,
    get_session_key_from_kwargs,
)
from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_session import session_to_nwb

data_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/_Raw_Data_for_future_reference")
published_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/000971")
output_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/fix_audit/misfiled_runs")
if output_dir_path.exists():
    shutil.rmtree(output_dir_path)
output_dir_path.mkdir(parents=True)

In [2]:
session_to_nwb_kwargs_per_session = fp_to_nwb(
    data_dir_path=data_dir_path, output_dir_path=output_dir_path, start_variable="Start Date", verbose=False
) + opto_to_nwb(data_dir_path=data_dir_path, output_dir_path=output_dir_path, start_variable="Start Date", verbose=False)


def find_sessions(subject_id: str, start_datetime: datetime) -> list[dict]:
    return [
        session_to_nwb_kwargs
        for session_to_nwb_kwargs in session_to_nwb_kwargs_per_session
        if session_to_nwb_kwargs["subject_id"] == subject_id
        and session_to_nwb_kwargs["session_conditions"].get("Start Date") == start_datetime.strftime("%m/%d/%y")
        and session_to_nwb_kwargs["session_conditions"].get("Start Time") == start_datetime.strftime("%H:%M:%S")
    ]

## Sessions whose own run is now converted

In [3]:
reconverted_sessions = [
    ("346.394", datetime(2020, 7, 31, 12, 3, 31)),
    ("344.400", datetime(2020, 7, 30, 12, 23, 54)),
]
selected_kwargs = []
for subject_id, start_datetime in reconverted_sessions:
    (session_to_nwb_kwargs,) = find_sessions(subject_id, start_datetime)
    selected_kwargs.append(session_to_nwb_kwargs)
no_port_entry_duration_sessions = get_no_port_entry_duration_sessions(
    port_entry_file_path=output_dir_path / "sessions_without_port_entry_durations.yaml",
    session_to_nwb_args_per_session=selected_kwargs,
)
for session_to_nwb_kwargs in selected_kwargs:
    if get_session_key_from_kwargs(session_to_nwb_kwargs) in no_port_entry_duration_sessions:
        session_to_nwb_kwargs["has_port_entry_durations"] = False
    session_to_nwb(**session_to_nwb_kwargs)

Reading port entry durations:   0%|          | 0/2 [00:00<?, ?it/s]

Reading port entry durations: 100%|██████████| 2/2 [00:00<00:00, 21.16it/s]

In [4]:
def summarize(file_path: Path) -> dict:
    with h5py.File(file_path, "r") as file:
        behavior = file["processing/behavior"]
        module_description = behavior.attrs["description"]
        summary = dict(
            msn=module_description.split("MSN = ")[1].split("\n")[0].strip(),
            box=module_description.split("Box = ")[1].strip(),
            session_description=file["session_description"][()].decode(),
        )
        for name in ("left_nose_poke_times", "right_nose_poke_times", "left_reward_times", "right_reward_times"):
            summary[name] = len(behavior[name]["timestamps"]) if name in behavior else 0
    return summary


rows = []
for subject_id, start_datetime in reconverted_sessions:
    session_stamp = start_datetime.isoformat().replace(":", "-")
    (published_file_path,) = (published_dir_path / f"sub-{subject_id.replace('.', '-')}").glob(f"*{session_stamp}*.nwb")
    (new_file_path,) = output_dir_path.glob(f"sub-{subject_id}_*{session_stamp}*.nwb")
    rows.append(dict(subject_id=subject_id, version="published", **summarize(published_file_path)))
    rows.append(dict(subject_id=subject_id, version="fixed", **summarize(new_file_path)))
pd.DataFrame(rows)

,subject_id,version,msn,box,session_description,left_nose_poke_times,right_nose_poke_times,left_reward_times,right_reward_times
0,346.394,published,FOOD_RI 60 RIGHT TTL,6,RI60 Training with concurrent fiber photometry...,4,823,0,50
1,346.394,fixed,Probe Test Habit Training TTL,5,Probe Test,0,23,0,0
2,344.400,published,Probe Test Habit Training TTL,2,Probe Test,7,0,0,0
3,344.400,fixed,RI 60 LEFT_STIM,1,"RI60 Training with optogenetic stimulation, re...",266,10,50,0


The raw runs at those start times, for reference:

In [5]:
raw_files = {
    "346.394": data_dir_path / "FP Experiments/Behavior/DPR/346.394/346.394",
    "344.400": data_dir_path / "Opto Experiments/DLS Excitatory/EYFP/344.400",
}
raw_rows = []
for subject_id, start_datetime in reconverted_sessions:
    medpc_variables = get_medpc_variables(
        file_path=raw_files[subject_id], variable_names=["Start Date", "Start Time", "Subject", "Box", "MSN"]
    )
    for start_date, start_time, subject_field, box, msn in zip(*medpc_variables.values()):
        if start_date == start_datetime.strftime("%m/%d/%y") and start_time == start_datetime.strftime("%H:%M:%S"):
            raw_rows.append(dict(file_owner=subject_id, subject_field=subject_field, box=box, msn=msn))
pd.DataFrame(raw_rows)

,file_owner,subject_field,box,msn
0,346.394,333,6,FOOD_RI 60 RIGHT TTL
1,346.394,346,5,Probe Test Habit Training TTL
2,344.400,271,2,Probe Test Habit Training TTL
3,344.400,344,1,RI 60 LEFT_STIM


## Published sessions that held another mouse's run

Each of these runs sits in the file of the mouse it was published under, but its `Subject:` field names a different mouse in the dataset. The fixed code no longer converts them.

In [6]:
removed_sessions = [
    ("113.283", datetime(2019, 7, 18, 12, 36, 46)),
    ("479", datetime(2019, 6, 13, 9, 43, 23)),
    ("322.415", datetime(2020, 10, 26, 11, 48, 23)),
    ("436.422", datetime(2020, 11, 13, 11, 37, 0)),
    ("123.411", datetime(2020, 11, 20, 13, 43, 3)),
    ("429.419", datetime(2020, 11, 20, 13, 43, 53)),
    ("343.483", datetime(2021, 10, 21, 15, 25, 37)),
    ("267.476", datetime(2021, 10, 28, 14, 23, 2)),
]
removed_rows = []
for subject_id, start_datetime in removed_sessions:
    session_stamp = start_datetime.isoformat().replace(":", "-")
    (published_file_path,) = (published_dir_path / f"sub-{subject_id.replace('.', '-')}").glob(f"*{session_stamp}*.nwb")
    published_summary = summarize(published_file_path)
    removed_rows.append(
        dict(
            published_file=published_file_path.name,
            published_msn=published_summary["msn"],
            published_box=published_summary["box"],
            sessions_in_fixed_code=len(find_sessions(subject_id, start_datetime)),
        )
    )
pd.DataFrame(removed_rows)

,published_file,published_msn,published_box,sessions_in_fixed_code
0,sub-113-283_ses-FP-PS-2019-07-18T12-36-46_beha...,RI 60 RIGHT STIM,1,0
1,sub-479_ses-Opto-DMS-Excitatory-ChR2-2019-06-1...,20sOmissions_TTL,2,0
2,sub-322-415_ses-Opto-DMS-Excitatory-ChR2Scramb...,FR1_LEFT_SCRAM,4,0
3,sub-436-422_ses-Opto-DMS-Inhibitory-NpHR-2020-...,RI 60 LEFT_STIM,2,0
4,sub-123-411_ses-Opto-DMS-Inhibitory-NpHR-2020-...,RI 60 LEFT_STIM,3,0
5,sub-429-419_ses-Opto-DMS-Excitatory-ChR2-2020-...,RI60_LEFT_SCRAM,4,0
6,sub-343-483_ses-Opto-DMS-Inhibitory-NpHRScramb...,RI 60 LEFT_STIM,3,0
7,sub-267-476_ses-Opto-DMS-Inhibitory-EYFP-2021-...,RI60_LEFT_SCRAM,1,0


Runs in those files at the published start times, with their `Subject:` fields:

In [7]:
raw_rows = []
for subject_id, start_datetime in removed_sessions:
    session_stamp = start_datetime.isoformat().replace(":", "-")
    for raw_file_path in sorted(
        {session_to_nwb_kwargs["behavior_file_path"] for session_to_nwb_kwargs in session_to_nwb_kwargs_per_session if session_to_nwb_kwargs["subject_id"] == subject_id}
    ):
        if raw_file_path.suffix.lower() == ".csv":
            continue
        medpc_variables = get_medpc_variables(
            file_path=raw_file_path, variable_names=["Start Date", "Start Time", "Subject", "Box", "MSN"]
        )
        for start_date, start_time, subject_field, box, msn in zip(*medpc_variables.values()):
            if start_date == start_datetime.strftime("%m/%d/%y") and start_time == start_datetime.strftime("%H:%M:%S"):
                raw_rows.append(
                    dict(file_owner=subject_id, start=session_stamp, subject_field=subject_field, box=box, msn=msn)
                )
pd.DataFrame(raw_rows)

,file_owner,start,subject_field,box,msn
0,113.283,2019-07-18T12-36-46,123.28,1,RI 60 RIGHT STIM
1,479,2019-06-13T09-43-23,480,2,20sOmissions_TTL
2,322.415,2020-10-26T11-48-23,321.415,4,FR1_LEFT_SCRAM
3,436.422,2020-11-13T11-37-00,435.422,2,RI 60 LEFT_STIM
4,123.411,2020-11-20T13-43-03,441.424,3,RI 60 LEFT_STIM
5,123.411,2020-11-20T13-43-03,128.424,2,RI60_RIGHT_SCRAM
6,123.411,2020-11-20T13-43-03,437.422,6,RI60_LEFT_SCRAM
7,429.419,2020-11-20T13-43-53,130.425,4,RI60_LEFT_SCRAM
8,343.483,2021-10-21T15-25-37,262.478,3,RI 60 LEFT_STIM
9,343.483,2021-10-21T15-25-37,268.476,5,FOOD_FR1 TTL Left
